In [ ]:
%load_ext autoreload
%autoreload 2

import os
import yaml
import polars as pl
import pandas as pd
import numpy as np
from tqdm import tqdm
from plotnine import *

from anngeno import AnnGeno

from scripts import get_correlations

import matplotlib.pyplot as plt

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs_absplice2.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

cov_list = config.get('covariates')

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list = list(set(all_annotation_list))
len(all_annotation_list)


## Functions

In [ ]:
import statsmodels.api as sm

# Get covariate corrected phenotypes
def cov_prs_correction(all_df, phenotypes, covariates=None, prs_pheno_map=None):
    # Initialize an empty DataFrame to store residuals
    # all_df.set_index('sample', inplace=True)
    cov_prs_corrected_phenos = pd.DataFrame(
        index=all_df.index
    )  # Index is the sample ID

    # Perform linear regression for each phenotype
    for pheno in tqdm(phenotypes):
        # Drop NaN values for the current phenotype
        if (prs_pheno_map is not None) and (covariates is not None):
            print(f"Correcting {pheno} with covariates and PRS")
            combined_df = all_df[[pheno] + covariates + [prs_pheno_map[pheno]]].dropna()
        elif covariates is not None:
            print(f"Correcting {pheno} with covariates only")
            combined_df = all_df[[pheno] + covariates].dropna()
        elif prs_pheno_map is not None:
            print(f"Correcting {pheno} with PRS only")
            combined_df = all_df[[pheno] + [prs_pheno_map[pheno]]].dropna()
        else:
            print(f"Not correcting {pheno}. Not enough information.")
            return all_df[[pheno]].dropna()
        y = combined_df[pheno]
        X = combined_df.drop(columns=[pheno])
        X = sm.add_constant(X)  # Add a constant term for the intercept

        # Fit the model
        model = sm.OLS(y, X).fit()

        # Save residuals
        residuals = pd.Series(model.resid, index=combined_df.index, name=pheno)
        cov_prs_corrected_phenos = pd.concat(
            [cov_prs_corrected_phenos, residuals], axis=1
        )

    # Reset the index for the resulting DataFrame
    cov_prs_corrected_phenos.reset_index(inplace=True)
    return cov_prs_corrected_phenos


In [ ]:
# Function to compute correlations
def compute_correlations_lazy(
    df_lazy: pl.LazyFrame,
    # filter_nan: bool = True,
) -> pl.LazyFrame:
    """
    Computes both Pearson and Spearman correlations:
    - Between ('sum', 'value'), ('max', 'value'), ('top2', 'value')
    - Grouped by ('annotation', 'phenotype')
    """
    # Remove individuals with no phenotype
    df_clean = df_lazy.filter(
        pl.col("value").is_not_nan() &
        pl.col("value").is_not_null() &
        pl.col("value").is_finite()
    )

    # Add ranks per group for Spearman
    df_ranks = df_clean.with_columns([
        pl.col("sum").rank().over(["annotation", "phenotype"]).alias("rank_sum"),
        pl.col("max").rank().over(["annotation", "phenotype"]).alias("rank_max"),
        pl.col("top2").rank().over(["annotation", "phenotype"]).alias("rank_top2"),
        pl.col("value").rank().over(["annotation", "phenotype"]).alias("rank_value"),
    ])

    # Group by + compute both sets of correlations
    correlations = df_ranks.group_by(["annotation", "phenotype"]).agg([
        # Pearson
        # pl.corr("sum", "value").alias("sum_pearson"),
        # pl.corr("max", "value").alias("max_pearson"),
        # pl.corr("top2", "value").alias("top2_pearson"),
        # Spearman
        pl.corr("rank_sum", "rank_value").alias("sum_spearman"),
        pl.corr("rank_max", "rank_value").alias("max_spearman"),
        pl.corr("rank_top2", "rank_value").alias("top2_spearman"),
    ])

    del df_ranks
    return correlations

def compute_gene_correlations(
    burdens_dir: str,
    pheno_corrected_df: pl.DataFrame,
    assocs_df: pl.DataFrame = None,
    config: dict = None,
    filter_nan: bool = True,
    subset_annos: list | None = None,
    subset_samples: list | None = None,
) -> pl.DataFrame:
    """
    Compute correlations between gene burden files and phenotypes,
    add annotation categories, fill NaNs, and return long-format correlations
    """
    # Prepare phenotype DataFrame
    pheno_df = pheno_corrected_df.unpivot(
        index=['sample_id'],
        variable_name='phenotype',
        value_name='value'
    )

    corr_df_list = []

    # for gene_file in tqdm(os.listdir(burdens_dir), desc="Correlations for genes"):
        # if not gene_file.endswith('.parquet'):
        #     continue
        # gene_id = gene_file.split('.')[0]
    for gene_id in tqdm(assocs_df['gene_id'].unique(), desc="Correlations for genes"):
        try:
            bdf = pl.scan_parquet(f'{burdens_dir}/{gene_id}.parquet')
        except FileNotFoundError:
            print(f"File for gene {gene_id} not found in {burdens_dir}. Skipping.")
            continue

        if subset_annos is not None:
            # print("Subsetting annotations")
            bdf = bdf.filter(pl.col('annotation').is_in(subset_annos))

        if subset_samples is not None:
            # print("Subsetting samples")
            bdf = bdf.filter(pl.col('sample_id').is_in(subset_samples))
        
        # Filter out or Fill rows with no variants in the gene
        if filter_nan:
            bdf_filtered = bdf.filter(
                pl.all_horizontal(
                    pl.col(['sum', 'max', 'top2']).is_not_nan()
                )
            )
        else:        
            # 1. Compute per-annotation mode for each column
            cols_to_fill = ['max', 'sum', 'top2']
            agg_exprs = [
                pl.col(col).drop_nans().mode().first().alias(f"{col}_mode")
                for col in cols_to_fill
            ]

            modes = bdf.group_by("annotation").agg(agg_exprs)

            # 2. Join the modes back
            bdf_with_modes = bdf.join(modes, on="annotation")

            # 3. Fill NaNs with group mode
            bdf_filtered = bdf_with_modes.with_columns([
                pl.when(pl.col(col).is_nan())
                .then(pl.col(f"{col}_mode"))
                .otherwise(pl.col(col))
                .alias(col)
                for col in cols_to_fill
            ]).drop([f"{col}_mode" for col in cols_to_fill])


        adf = assocs_df.filter(pl.col('gene_id') == gene_id)
        phenos_needed = adf['phenotype'].unique().to_list()

        pheno_filtered = (
            pheno_df
            .filter(pl.col('phenotype').is_in(phenos_needed))
            .lazy()
        )

        cdf = bdf_filtered.join(pheno_filtered, on='sample_id', how='left')

        corr_df_list.append(
            compute_correlations_lazy(cdf).with_columns(
                pl.lit(gene_id).alias('gene_id'),
            ).collect()
        )

    corr_df = pl.concat(corr_df_list)

    # Annotations categories
    rare_variant_annotations_dict = config.get('rare_variant_annotations')
    rare_variant_annotations_dict = {
        k: v for k, v in rare_variant_annotations_dict.items() if k != 'misc'
    } if rare_variant_annotations_dict else {}

    # Build annotation -> category map
    annotation_category_map = {}
    for category, annotations in rare_variant_annotations_dict.items():
        for ann in annotations:
            annotation_category_map[ann] = category

    # Add category column
    corr_df = corr_df.with_columns(
        pl.col("annotation").replace(annotation_category_map).alias("category")
    )

    # Fill NaNs in correlation columns
    # corr_cols = [col for col in corr_df.columns if "pearson" in col or "spearman" in col]
    # corr_df = corr_df.with_columns(
    #     [pl.col(col).fill_null(0).alias(col) for col in corr_cols]
    # )

    # return corr_df

    # Melt to long format
    corr_long = corr_df.unpivot(
        index=["annotation", "phenotype", "gene_id", "category"],
        variable_name="correlation_type",
        value_name="correlation"
    ).with_columns([
        pl.col("correlation_type").str.extract(r"(pearson|spearman)").alias("method"),
        pl.col("correlation_type").str.extract(r"(sum|max|top2)").alias("aggregation"),
        # pl.col("correlation").abs().alias("abs_correlation")
    ])

    del corr_df
    return corr_long

## Pheno GIS plot

In [ ]:

def pheno_gene_plot_data(
    config_path: str,
    phenotype: str,
    gene_id: str,
    annotation: str,
    pheno_df_path: str,
    gene_burdens_path: str,
    filter_nan: bool = False,
    subset_samples: list[str] = None,
):
    """
    Loads config, corrections, associations, burdens, and joins them.
    """

    # Load config
    with open(config_path) as f:
        config = yaml.safe_load(f)
    covs = config.get("covariates")

    # Load and correct phenotype data
    pheno_list = [phenotype + '_prs_corrected']
    pheno_df = pl.read_parquet(pheno_df_path, columns=["eid"] + covs + pheno_list).rename({'eid': 'sample_id'})
    pheno_corrected_df = pl.from_pandas(
        cov_prs_correction(
            pheno_df.to_pandas().set_index('sample_id'),
            pheno_list,
            covs
        )
    )
    pheno_corrected_df.columns = ['sample_id'] + [phenotype]

    # Unpivot to long format
    pheno_long_df = pheno_corrected_df.unpivot(
        index=['sample_id'],
        variable_name='phenotype',
        value_name='value'
    ).lazy()


    # Load burdens file for gene
    try:
        bdf = pl.scan_parquet(gene_burdens_path)
    except FileNotFoundError:
        print(f"File not found at {gene_burdens_path}. Skipping.")
        return None, None

    bdf = bdf.filter(pl.col('annotation') == annotation)

    # Optional filtering
    if subset_samples is not None:
        bdf = bdf.filter(pl.col('sample_id').is_in(subset_samples))

    if filter_nan:
        bdf_filtered = bdf.filter(
            pl.all_horizontal(
                pl.col(['sum', 'max', 'top2']).is_not_nan()
            )
        )
    else:        
        # 1. Compute per-annotation mode for each column
        cols_to_fill = ['max', 'sum', 'top2']
        agg_exprs = [
            pl.col(col).drop_nans().mode().first().alias(f"{col}_mode")
            for col in cols_to_fill
        ]

        modes = bdf.group_by("annotation").agg(agg_exprs)

        # 2. Join the modes back
        bdf_with_modes = bdf.join(modes, on="annotation")

        # 3. Fill NaNs with group mode
        bdf_filtered = bdf_with_modes.with_columns([
            pl.when(pl.col(col).is_nan())
            .then(pl.col(f"{col}_mode"))
            .otherwise(pl.col(col))
            .alias(col)
            for col in cols_to_fill
        ]).drop([f"{col}_mode" for col in cols_to_fill])


    # Join burdens and phenotypes
    cdf = bdf_filtered.join(pheno_long_df, on='sample_id', how='left').collect()

    return pheno_long_df.collect(), bdf.collect(), cdf


In [ ]:
pheno = 'hdl_cholesterol'
gene_id = 'ENSG00000073060' 
anno = 'am_pathogenicity' #'AbSplice2_max'
# eur_samples = pl.read_parquet('/home/dnanexus/data_dir/167k_sample_ids.parquet')['sample_id'].to_list()

_, _, pg_plot = pheno_gene_plot_data(
    config_path = '/home/dnanexus/ukbgym/config_wgs_absplice2.yaml',
    phenotype = pheno,
    gene_id = gene_id,
    annotation = anno,
    pheno_df_path = '/home/dnanexus/data_dir/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet',
    gene_burdens_path = f'/home/dnanexus/absplice2_61genes/{gene_id}.parquet',
    # subset_samples = eur_samples,
)

pg_plot

In [ ]:
aggregation = "top2"

(
    ggplot(pg_plot, aes(x=aggregation, y='value')) +
    geom_point(alpha=0.25) +
    geom_smooth(method='lm', se=True, color='red') +
    labs(x=f'{anno}',
         y=pheno) +
    theme_bw() +
    theme(
        figure_size=(4, 5),
        axis_title=element_text(size=16)
    )
)

## Compute correlations

In [ ]:
a = pl.read_parquet('/home/dnanexus/data_dir/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet')
t = [trait[:-14] for trait in a.columns if trait.endswith('_prs_corrected')]

ags = os.listdir('/home/dnanexus/absplice2_61genes')
ags = [ag.split('.')[0] for ag in ags if ag.endswith('.parquet')]

assocs = pl.read_parquet('/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq')
assocs = assocs.with_columns(
    pl.col("description").str.to_lowercase().str.replace_all(" ", "_").alias("phenotype")
).filter(
    pl.col('annotation').str.contains("pLoF")
)

assocs = assocs.filter(pl.col('gene_id').is_in(ags) & pl.col('phenotype').is_in(t))
# assocs.write_parquet('/home/dnanexus/data_dir/absplice2_assocs_expanded.parquet')

In [ ]:
config_path = f'/home/dnanexus/ukbgym/config_wgs_absplice2.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

covs = config.get("covariates")

data_dir = '/home/dnanexus/data_dir'
associations_df_path = f'{data_dir}/absplice2_assocs.parquet'
pheno_df_path = f'{data_dir}/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'

assocs = pl.read_parquet(associations_df_path)
phenotypes = list(assocs['phenotype'].unique())
pheno_list = [p + '_prs_corrected' for p in phenotypes]
pheno_df = pl.read_parquet(pheno_df_path, columns=["eid"] + covs + pheno_list).rename({'eid': 'sample_id'})

pheno_corrected_df = pl.from_pandas(cov_prs_correction(pheno_df.to_pandas().set_index('sample_id'), pheno_list, covs))

# prs_file = f'{data_dir}/PRS.parquet'
# prs_pheno_map_file = f'{data_dir}/prs_pheno_map_clean.csv'
# prs_pheno_map = pd.read_csv(prs_pheno_map_file)
# prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
# prs_df = pl.read_parquet(prs_file).rename({'sample': 'sample_id'})

# all_df = pheno_df.join(prs_df, on='sample_id', how='inner').to_pandas()
# pheno_corrected_df = pl.from_pandas(cov_prs_correction(all_df.set_index('sample_id'), phenotypes, covs, prs_pheno_map))

pheno_corrected_df.columns = ['sample_id'] + phenotypes
pheno_corrected_df

In [ ]:
# Read associations
assocs_df = pl.read_parquet('/home/dnanexus/data_dir/absplice2_assocs.parquet')

eur_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv').with_columns(
    pl.col('eid').cast(pl.Utf8).alias('sample_id')
)['sample_id'].to_list()

# eur_samples = pl.read_parquet('/home/dnanexus/data_dir/167k_sample_ids.parquet')['sample_id'].to_list()

corr_long = compute_gene_correlations(
    burdens_dir='/home/dnanexus/absplice2_61genes',
    pheno_corrected_df=pheno_corrected_df,
    assocs_df=assocs_df,
    config=config,
    filter_nan=False,
    subset_annos=['loftee_hc', 'am_pathogenicity', 'pangolin_score', 'AbSplice2_max'],
    subset_samples=eur_samples,
)

corr_long

## Make plots

In [ ]:
dir_df = corr_long.filter(pl.col('annotation')=='loftee_hc').filter(pl.col('aggregation')=='max').pivot(
    index=['annotation', 'phenotype', 'gene_id'],
    on='aggregation',
    values='correlation'
)

# Make sure all aggregations point in the same direction
# dir_df.filter(
#     ((pl.col('sum')>0) & (pl.col('max')>0) & (pl.col('top2')>0)) |
#     ((pl.col('sum')<0) & (pl.col('max')<0) & (pl.col('top2')<0))
# )

dir_df = dir_df.select(['phenotype', 'gene_id', 'max']).with_columns(
    (pl.col('max')).alias('lof_correlation'),
    (pl.col('max')/pl.col('max').abs()).alias('lof_direction')
).drop(['max'])
dir_df

In [ ]:
plt_df = corr_long.join(
    dir_df,
    on=['phenotype', 'gene_id'],
    how='left'
).with_columns(
    (pl.col('correlation') * pl.col('lof_direction')).alias('new_correlation'),
    (pl.col('correlation')/pl.col('lof_correlation')).alias('scaled_correlation')
)

plt_df

In [ ]:
aggregation = "max"
method = "spearman"


agg_df = (
    plt_df
    .filter(pl.col("aggregation") == aggregation)
    .filter(pl.col("method") == method)
    .group_by("annotation")
    .agg(pl.median("new_correlation").alias("median_new_correlation"))
    .sort("median_new_correlation", descending=True)
)

ordered_annotations = agg_df['annotation'].to_list()

corr_long_pd = plt_df.to_pandas()
# corr_long_pd = corr_long.filter(pl.col('annotation').is_in(subset_annos)).to_pandas()
corr_long_pd['annotation'] = pd.Categorical(
    corr_long_pd['annotation'],
    categories=ordered_annotations,
    ordered=True
)

(
    ggplot(
        corr_long_pd.query(f"aggregation == '{aggregation}' & method == '{method}'"),
        aes(x='annotation', y='new_correlation', fill='category')
    )
    + geom_boxplot(alpha=0.75)
    + theme_bw()
    # + scale_y_sqrt()
    + ylab('rank correlation')
    + theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(8, 6),
    )
)

In [ ]:
aggregation = "max"
method = "spearman"


agg_df = (
    plt_df
    .filter(pl.col("aggregation") == aggregation)
    .filter(pl.col("method") == method)
    .group_by("annotation")
    .agg(pl.median("scaled_correlation").alias("median_scaled_correlation"))
    .sort("median_scaled_correlation", descending=True)
)

ordered_annotations = agg_df['annotation'].to_list()

corr_long_pd = plt_df.to_pandas()
# corr_long_pd = corr_long.filter(pl.col('annotation').is_in(subset_annos)).to_pandas()
corr_long_pd['annotation'] = pd.Categorical(
    corr_long_pd['annotation'],
    categories=ordered_annotations,
    ordered=True
)

(
    ggplot(
        corr_long_pd.query(f"aggregation == '{aggregation}' & method == '{method}'"),
        aes(x='annotation', y='scaled_correlation', fill='category')
    )
    + geom_boxplot(alpha=0.75)
    + theme_bw()
    # + scale_y_sqrt()
    + ylab('scaled rank correlation')
    + theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(8, 6),
    )
)

In [ ]:
aggregation = "max"
method = "spearman"


agg_df = (
    plt_df
    .filter(pl.col("aggregation") == aggregation)
    .filter(pl.col("method") == method)
    .group_by("annotation")
    .agg(pl.median("abs_correlation").alias("median_abs_correlation"))
    .sort("median_abs_correlation", descending=True)
)

ordered_annotations = agg_df['annotation'].to_list()

corr_long_pd = plt_df.to_pandas()
# corr_long_pd = corr_long.filter(pl.col('annotation').is_in(subset_annos)).to_pandas()
corr_long_pd['annotation'] = pd.Categorical(
    corr_long_pd['annotation'],
    categories=ordered_annotations,
    ordered=True
)

(
    ggplot(
        corr_long_pd.query(f"aggregation == '{aggregation}' & method == '{method}'"),
        aes(x='annotation', y='abs_correlation', fill='category')
    )
    + geom_boxplot(alpha=0.75)
    + theme_bw()
    # + scale_y_sqrt()
    + ylab('|rank correlation|')
    + theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(8, 6),
    )
)

In [ ]:
corr_long.filter(pl.col('correlation') > 0.025).filter(pl.col('method')=='spearman')

## Debug script

In [ ]:
%load_ext autoreload
%autoreload 2

from scripts import get_correlations

config_path='/home/dnanexus/ukbgym/config_wgs_absplice2.yaml'
burdens_dir='/home/dnanexus/absplice2_61genes'
associations_file='/home/dnanexus/data_dir/absplice2_assocs.parquet'
pheno_file='/home/dnanexus/data_dir/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'
subset_annos=['loftee_hc', 'am_pathogenicity', 'pangolin_score', 'AbSplice2_max']

subset_samples = pl.read_csv('/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv').with_columns(
    pl.col('eid').cast(pl.Utf8).alias('sample_id')
)['sample_id'].to_list()

with open(config_path) as f:
    config = yaml.safe_load(f)

covs = config.get("covariates")

assocs_df = pl.read_parquet(associations_file)
phenotypes = list(assocs_df['phenotype'].unique())
pheno_list = [p + '_prs_corrected' for p in phenotypes]

pheno_df = pl.read_parquet(pheno_file, columns=["eid"] + covs + pheno_list).rename({'eid': 'sample_id'})

corrected_df = cov_prs_correction(
    pheno_df.to_pandas().set_index('sample_id'),
    pheno_list,
    covariates=covs
)
pheno_corrected_df = pl.from_pandas(corrected_df)
pheno_corrected_df.columns = ['sample_id'] + phenotypes

corr_long = get_correlations.compute_correlations(
    burdens_dir=burdens_dir,
    pheno_corrected_df=pheno_corrected_df,
    assocs_df=assocs_df,
    config=config,
    filter_nan=False,
    subset_annos=subset_annos,
    subset_samples=subset_samples,
)

corr_long

In [ ]:
dir_df = corr_long.filter(pl.col('annotation')=='loftee_hc').filter(pl.col('aggregation')=='max').pivot(
    index=['annotation', 'phenotype', 'gene_id'],
    on='aggregation',
    values='correlation'
)

dir_df = dir_df.select(['phenotype', 'gene_id', 'max']).with_columns(
    (pl.col('max')).alias('lof_correlation'),
    (pl.col('max')/pl.col('max').abs()).alias('lof_direction')
).drop(['max'])

plt_df = corr_long.join(
    dir_df,
    on=['phenotype', 'gene_id'],
    how='left'
).with_columns(
    (pl.col('correlation') * pl.col('lof_direction')).alias('new_correlation'),
    (pl.col('correlation')/pl.col('lof_correlation')).alias('scaled_correlation')
)

plt_df

In [ ]:
aggregation = "max"
method = "spearman"


agg_df = (
    plt_df
    .filter(pl.col("aggregation") == aggregation)
    .filter(pl.col("method") == method)
    .group_by("annotation")
    .agg(pl.median("new_correlation").alias("median_new_correlation"))
    .sort("median_new_correlation", descending=True)
)

ordered_annotations = agg_df['annotation'].to_list()

corr_long_pd = plt_df.to_pandas()
# corr_long_pd = corr_long.filter(pl.col('annotation').is_in(subset_annos)).to_pandas()
corr_long_pd['annotation'] = pd.Categorical(
    corr_long_pd['annotation'],
    categories=ordered_annotations,
    ordered=True
)

(
    ggplot(
        corr_long_pd.query(f"aggregation == '{aggregation}' & method == '{method}'"),
        aes(x='annotation', y='new_correlation', fill='category')
    )
    + geom_boxplot(alpha=0.75)
    + theme_bw()
    # + scale_y_sqrt()
    + ylab('rank correlation')
    + theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(8, 6),
    )
)